# MMFDB standalone — no lock-in, combine any libraries

MMFDB is a **standalone package**: it does not need ChiSurf, and ChiSurf is an
*optional* consumer of it. This notebook uses MMFDB entirely on its own to track
the provenance of **two different third-party libraries** — `tttrlib` and
`FRETBursts` — analysing the same measurement. Because MMFDB only records *what
a tool did* (not *which* tool), there is no lock-in: mix and match whatever
analysis software you like.

> Notice there is **no `import chisurf`** anywhere below — the last cell asserts
> it was never imported.

In [1]:
import contextlib, hashlib, io, sys, tempfile, warnings
from pathlib import Path
import numpy as np
warnings.filterwarnings("ignore")
# FRETBursts 0.8.3 predates NumPy 2; restore the one symbol it needs.
if not hasattr(np, "asfarray"):
    np.asfarray = lambda a, dtype=np.float64: np.asarray(a, dtype=dtype)

import tttrlib                          # analysis library #1
from mmfdb.config import configure_runtime
from mmfdb.repository import MFDatabase  # MMFDB, used directly — no server, no chisurf

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

## 1. Open a standalone MMFDB and register a measurement

`MFDatabase` is the embedded interface — a local SQLite-backed store, no server
required. We simulate a two-state smFRET measurement with `tttrlib` and register
the raw file as a provenance artifact.

In [2]:
work = Path(tempfile.mkdtemp())
configure_runtime(database_path=work / "mmfdb.db", settings_dir=work)
DB = str(work / "mmfdb.db")

cfg = {"settings": {"dt": 0.01, "n_ph_max": 900_000, "n_channels": 2, "laser_period": 32.0,
        "fast_grid_bbox": True, "active_margin": 1.0, "seed_diffusion": 1, "seed_emission": 2},
       "box": {"xy": 2.0, "z": 4.0},
       "species": [{"D": 0.05, "q": [225.0, 75.0]}, {"D": 0.05, "q": [75.0, 225.0]}],
       "k_rad": [0, 0, 0, 0], "k_nrad": [0.0, 2.0, 2.0, 0.0],
       "background": [0.5, 0.5], "population": [0.25, 0.25],
       "excitation": {"type": "gaussian3d", "w0": 0.3, "z0": 2.0,
                      "extent_xy": 2.0, "extent_z": 4.0, "spacing": 0.1}}
engine = tttrlib.SimEngine.from_dict(cfg); engine.run()
raw = work / "sim.spc"; engine.to_tttr(dt=0.01, n_channels=2, laser_period=32.0).write(str(raw))

with MFDatabase(DB) as db:
    db.register_artifact(artifact_id="raw/sim", artifact_kind="raw_measurement",
                         file_path=str(raw), checksum=sha256(raw), size_bytes=raw.stat().st_size)
print("registered raw measurement in MMFDB:", raw.name)

registered raw measurement in MMFDB: sim.spc


A one-line helper records a tool run: it links `raw → operation → result` and
stores the tool's name, version, and parameters.

In [3]:
def record_tool(op_id, software, version, params, out_id, out_path):
    with MFDatabase(DB) as db:
        db.record_operation_with_artifacts(
            operation_id=op_id, operation_type="burst_selection", status="succeeded",
            software_package=software, software_version=version,
            input_artifacts=[{"artifact_id": "raw/sim", "artifact_kind": "raw_measurement",
                              "file_path": str(raw), "checksum": sha256(raw), "role": "raw"}],
            output_artifacts=[{"artifact_id": out_id, "artifact_kind": "burst_table",
                               "file_path": str(out_path), "checksum": sha256(out_path), "role": "bursts"}],
            parameters=[{"parameter_uuid": f"{op_id}/{k}", "name": k, "value": v}
                        for k, v in params.items()])

## 2. Library #1 — `tttrlib`

Run tttrlib's own burst search and record it.

In [4]:
data = tttrlib.TTTR(str(raw), "SPC-130")
ss = np.asarray(data.burst_search_cusum_sprt(min_photons=20, background_cps=500.0,
                signal_to_background_ratio=4.0, alpha=0.01, beta=0.01))
rout = np.asarray(data.routing_channels)
pr = [int((rout[int(a):int(z)+1] == 1).sum()) / max(1, int((rout[int(a):int(z)+1] != -1).sum()))
      for a, z in zip(ss[0::2], ss[1::2])]
out_a = work / "bursts_tttrlib.csv"; np.savetxt(out_a, np.array(pr), header="PR", comments="")
record_tool("op/tttrlib", "tttrlib", tttrlib.__version__, {"min_photons": 20}, "res/tttrlib", out_a)
print(f"tttrlib: {len(pr)} bursts, mean E = {np.mean(pr):.3f}")

tttrlib: 3599 bursts, mean E = 0.501


## 3. Library #2 — FRETBursts

A completely different package ([FRETBursts](https://fretbursts.readthedocs.io))
analyses the **same** raw measurement, via the Photon-HDF5 interchange format, and
is recorded the same way — no MMFDB change needed to support it.

In [5]:
import phconvert as phc
macro = np.asarray(data.macro_times).astype("int64"); det = np.asarray(data.routing_channels).astype("uint8")
res = float(data.header.macro_time_resolution); total = macro[-1] * res
h5 = work / "fb.hdf5"
phc.hdf5.save_photon_hdf5(dict(description="sim", identity=dict(author="sim"),
    setup=dict(num_pixels=2, num_spots=1, num_spectral_ch=2, num_polarization_ch=1, num_split_ch=1,
        modulated_excitation=False, lifetime=False, excitation_alternated=[False], excitation_cw=[True],
        excitation_wavelengths=[532e-9], detection_wavelengths=[580e-9, 640e-9]),
    photon_data=dict(timestamps=macro, detectors=det, timestamps_specs=dict(timestamps_unit=res),
        measurement_specs=dict(measurement_type="smFRET",
            detectors_specs=dict(spectral_ch1=np.array([0], "uint8"), spectral_ch2=np.array([1], "uint8"))))),
    h5_fname=str(h5), overwrite=True)

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    import fretbursts as fb
    from fretbursts import loader
    d = loader.photon_hdf5(str(h5))
    d.calc_bg(fb.bg.exp_fit, time_s=max(total / 3, 1.0), tail_min_us=200)
    d.burst_search(F=6, m=10)
    ds = d.select_bursts(fb.select_bursts.size, th1=20)
E = np.asarray(ds.E[0]); out_b = work / "bursts_fretbursts.csv"; np.savetxt(out_b, E, header="E", comments="")
record_tool("op/fretbursts", "FRETBursts", fb.__version__, {"F": 6, "m": 10}, "res/fretbursts", out_b)
print(f"FRETBursts: {int(ds.num_bursts[0])} bursts, mean E = {np.nanmean(E):.3f}")

Saving: /var/folders/cl/txwq_hq52rl_t0f9xjk3g5c00000gn/T/tmpp9cff64a/fb.hdf5


FRETBursts: 3229 bursts, mean E = 0.498


## 4. One lineage, two libraries, no ChiSurf

Both tools now hang off the same raw measurement in the provenance graph.

In [6]:
with MFDatabase(DB) as db:
    graph = db.export_provenance_graph("artifact", "raw/sim")
print(f"provenance: {len(graph['nodes'])} nodes, {len(graph['edges'])} edges")
for e in graph["edges"]:
    print(f"  {e['source_node_id']:16s} --{e['relationship_type']}--> {e['target_node_id']}")

assert "chisurf" not in sys.modules, "chisurf should never be imported here"
print("\n\u2713 MMFDB tracked two independent libraries — and ChiSurf was never imported.")

provenance: 5 nodes, 4 edges
  raw/sim          --input_to--> op/tttrlib
  raw/sim          --input_to--> op/fretbursts
  op/tttrlib       --produced--> res/tttrlib
  op/fretbursts    --produced--> res/fretbursts

✓ MMFDB tracked two independent libraries — and ChiSurf was never imported.


## Takeaway

MMFDB is usable on its own (`pip install` the `mmfdb` package, `import mmfdb`) and
records the provenance of *any* tool by name and version. ChiSurf is one optional
consumer among many — your pipeline can freely combine `tttrlib`, FRETBursts, a
shell script, or your own code, and MMFDB keeps them all traceable to the raw data.